# Python practice 9: dates, times and time zones

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-practice-09-dates-times-and-time-zones.ipynb)

13 exercises on date arithmetic, time zones and DST, parsing and formatting, and calendar logic. Each exercise names the [reference entries](../learning/python-reference-09-dates-times-and-time-zones.ipynb) it practises.

Write each function in the empty cell under its exercise and compare your results with the sample calls. Open the solution cell when you are done or stuck. 6 easy, 6 medium, 1 hard.

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Parse mixed timestamp formats](#exercise-1-parse-mixed-timestamp-formats) | Medium | [ISO 8601, `strftime` and `strptime`](../learning/python-reference-09-dates-times-and-time-zones.ipynb#iso-8601-strftime-and-strptime) |
| 2 | [Local wall time to UTC](#exercise-2-local-wall-time-to-utc) | Easy | [Naive and aware datetimes](../learning/python-reference-09-dates-times-and-time-zones.ipynb#naive-and-aware-datetimes) |
| 3 | [Hours in a local day](#exercise-3-hours-in-a-local-day) | Medium | [Daylight saving time](../learning/python-reference-09-dates-times-and-time-zones.ipynb#daylight-saving-time) |
| 4 | [Elapsed versus wall-clock addition](#exercise-4-elapsed-versus-wall-clock-addition) | Medium | [Daylight saving time](../learning/python-reference-09-dates-times-and-time-zones.ipynb#daylight-saving-time) |
| 5 | [Month boundaries](#exercise-5-month-boundaries) | Easy | [Month ends, adding months and date ranges](../learning/python-reference-09-dates-times-and-time-zones.ipynb#month-ends-adding-months-and-date-ranges) |
| 6 | [Add months with clamping](#exercise-6-add-months-with-clamping) | Medium | [Month ends, adding months and date ranges](../learning/python-reference-09-dates-times-and-time-zones.ipynb#month-ends-adding-months-and-date-ranges) |
| 7 | [Business days between dates](#exercise-7-business-days-between-dates) | Medium | [`date`, `time`, `datetime`, `timedelta`](../learning/python-reference-09-dates-times-and-time-zones.ipynb#date-time-datetime-timedelta) |
| 8 | [Fill gaps in a daily series](#exercise-8-fill-gaps-in-a-daily-series) | Medium | [Month ends, adding months and date ranges](../learning/python-reference-09-dates-times-and-time-zones.ipynb#month-ends-adding-months-and-date-ranges) |
| 9 | [Bucket events into 15-minute windows](#exercise-9-bucket-events-into-15-minute-windows) | Easy | [Month ends, adding months and date ranges](../learning/python-reference-09-dates-times-and-time-zones.ipynb#month-ends-adding-months-and-date-ranges) |
| 10 | [Epoch milliseconds](#exercise-10-epoch-milliseconds) | Easy | [Unix timestamps](../learning/python-reference-09-dates-times-and-time-zones.ipynb#unix-timestamps) |
| 11 | [Human-readable durations](#exercise-11-human-readable-durations) | Easy | [`date`, `time`, `datetime`, `timedelta`](../learning/python-reference-09-dates-times-and-time-zones.ipynb#date-time-datetime-timedelta) |
| 12 | [ISO week labels](#exercise-12-iso-week-labels) | Easy | [`date`, `time`, `datetime`, `timedelta`](../learning/python-reference-09-dates-times-and-time-zones.ipynb#date-time-datetime-timedelta) |
| 13 | [Cut-off times in local zones](#exercise-13-cut-off-times-in-local-zones) | Hard | [Naive and aware datetimes](../learning/python-reference-09-dates-times-and-time-zones.ipynb#naive-and-aware-datetimes), [Daylight saving time](../learning/python-reference-09-dates-times-and-time-zones.ipynb#daylight-saving-time) |

*Last reviewed: November 2026, against Python 3.13.*

In [ ]:
# Imports used by the sample calls and the solutions.
import io
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from fractions import Fraction

## Exercise 1: Parse mixed timestamp formats

**Difficulty:** Medium · about 10 min · **Practises:** [ISO 8601, `strftime` and `strptime`](../learning/python-reference-09-dates-times-and-time-zones.ipynb#iso-8601-strftime-and-strptime)

*Sources rarely agree on a timestamp format; ingestion normalises them.*

Write `normalise(text)` returning an ISO string in UTC with seconds precision (`2026-11-27T09:30:00+00:00`) from ISO 8601 (with `Z` or an offset), `DD/MM/YYYY HH:MM` (assume UTC) or epoch seconds as digits. Raise `ValueError` for anything else.

**Sample calls**

| call | expected |
| --- | --- |
| `normalise("2026-11-27T09:30:00Z")` | `'2026-11-27T09:30:00+00:00'` |
| `normalise("2026-11-27T15:00:00+05:30")` | `'2026-11-27T09:30:00+00:00'` |
| `normalise("27/11/2026 09:30")` | `'2026-11-27T09:30:00+00:00'` |

In [ ]:
def normalise(text):
    # your code here
    pass

In [ ]:
from datetime import datetime, timezone

def normalise(text):
    text = text.strip()
    if text.isdigit():
        dt = datetime.fromtimestamp(int(text), tz=timezone.utc)
    else:
        try:
            dt = datetime.fromisoformat(text)
        except ValueError:
            dt = datetime.strptime(text, "%d/%m/%Y %H:%M")
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).isoformat(timespec="seconds")

## Exercise 2: Local wall time to UTC

**Difficulty:** Easy · about 5 min · **Practises:** [Naive and aware datetimes](../learning/python-reference-09-dates-times-and-time-zones.ipynb#naive-and-aware-datetimes)

*Business events are recorded in local time and must be stored in UTC.*

Write `to_utc(local_iso, zone)` taking a naive ISO string and an IANA zone name and returning the UTC ISO string.

**Sample calls**

| call | expected |
| --- | --- |
| `to_utc("2026-01-15T09:00:00", "America/New_York")` | `'2026-01-15T14:00:00+00:00'` |
| `to_utc("2026-07-15T09:00:00", "America/New_York")` | `'2026-07-15T13:00:00+00:00'` |
| `to_utc("2026-11-27T09:00:00", "Asia/Kolkata")` | `'2026-11-27T03:30:00+00:00'` |

In [ ]:
def to_utc(local_iso, zone):
    # your code here
    pass

In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

def to_utc(local_iso, zone):
    return datetime.fromisoformat(local_iso).replace(tzinfo=ZoneInfo(zone)).astimezone(timezone.utc).isoformat()

## Exercise 3: Hours in a local day

**Difficulty:** Medium · about 10 min · **Practises:** [Daylight saving time](../learning/python-reference-09-dates-times-and-time-zones.ipynb#daylight-saving-time)

*Hourly partitions per local day: some days have 23 or 25 hours, a classic source of missing or duplicate data.*

Write `local_hours(day, zone)` returning the number of real hours in that local calendar day (a `date`).

**Sample calls**

| call | expected |
| --- | --- |
| `local_hours(date(2026, 11, 27), "America/New_York")` | `24.0` |
| `local_hours(date(2026, 3, 8), "America/New_York")` | `23.0` |
| `local_hours(date(2026, 11, 1), "America/New_York")` | `25.0` |

In [ ]:
def local_hours(day, zone):
    # your code here
    pass

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

def local_hours(day, zone):
    tz = ZoneInfo(zone)
    start = datetime(day.year, day.month, day.day, tzinfo=tz)
    nxt = day + timedelta(days=1)
    end = datetime(nxt.year, nxt.month, nxt.day, tzinfo=tz)
    return (end.astimezone(timezone.utc) - start.astimezone(timezone.utc)) / timedelta(hours=1)

## Exercise 4: Elapsed versus wall-clock addition

**Difficulty:** Medium · about 10 min · **Practises:** [Daylight saving time](../learning/python-reference-09-dates-times-and-time-zones.ipynb#daylight-saving-time)

*Schedulers must decide whether "every 24 hours" means the same local time or exactly 24 hours.*

Write `add_hours(start_iso, zone, hours, mode)` taking a naive local ISO string; `mode` is `"wall"` (add to the local time) or `"elapsed"` (add real hours); return the resulting local ISO string with offset.

**Sample calls**

| call | expected |
| --- | --- |
| `add_hours("2026-03-07T12:00:00", "America/New_York", 24, "wall")` | `'2026-03-08T12:00:00-04:00'` |
| `add_hours("2026-03-07T12:00:00", "America/New_York", 24, "elapsed")` | `'2026-03-08T13:00:00-04:00'` |
| `add_hours("2026-06-01T12:00:00", "America/New_York", 24, "elapsed")` | `'2026-06-02T12:00:00-04:00'` |

In [ ]:
def add_hours(start_iso, zone, hours, mode):
    # your code here
    pass

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

def add_hours(start_iso, zone, hours, mode):
    tz = ZoneInfo(zone)
    start = datetime.fromisoformat(start_iso).replace(tzinfo=tz)
    if mode == "wall":
        result = start + timedelta(hours=hours)
    elif mode == "elapsed":
        result = (start.astimezone(timezone.utc) + timedelta(hours=hours)).astimezone(tz)
    else:
        raise ValueError(f"unknown mode {mode!r}")
    return result.isoformat()

## Exercise 5: Month boundaries

**Difficulty:** Easy · about 5 min · **Practises:** [Month ends, adding months and date ranges](../learning/python-reference-09-dates-times-and-time-zones.ipynb#month-ends-adding-months-and-date-ranges)

*Monthly reports need the first and last day of the month.*

Write `month_bounds(d)` returning `(first_day, last_day)` of `d`'s month as ISO strings.

**Sample calls**

| call | expected |
| --- | --- |
| `month_bounds(date(2026, 11, 27))` | `('2026-11-01', '2026-11-30')` |
| `month_bounds(date(2028, 2, 10))` | `('2028-02-01', '2028-02-29')` |
| `month_bounds(date(2026, 2, 28))` | `('2026-02-01', '2026-02-28')` |

In [ ]:
def month_bounds(d):
    # your code here
    pass

In [ ]:
import calendar

def month_bounds(d):
    last = calendar.monthrange(d.year, d.month)[1]
    return d.replace(day=1).isoformat(), d.replace(day=last).isoformat()

## Exercise 6: Add months with clamping

**Difficulty:** Medium · about 10 min · **Practises:** [Month ends, adding months and date ranges](../learning/python-reference-09-dates-times-and-time-zones.ipynb#month-ends-adding-months-and-date-ranges)

*Billing dates and monthly schedules move by months, not by 30 days.*

Write `add_months(d, months)` returning the date `months` later (or earlier), clamping the day to the target month's length. Return an ISO string.

**Sample calls**

| call | expected |
| --- | --- |
| `add_months(date(2026, 1, 31), 1)` | `'2026-02-28'` |
| `add_months(date(2028, 1, 31), 1)` | `'2028-02-29'` |
| `add_months(date(2026, 11, 30), 3)` | `'2027-02-28'` |

In [ ]:
def add_months(d, months):
    # your code here
    pass

In [ ]:
import calendar

def add_months(d, months):
    index = d.month - 1 + months
    year, month = d.year + index // 12, index % 12 + 1
    return d.replace(year=year, month=month, day=min(d.day, calendar.monthrange(year, month)[1])).isoformat()

## Exercise 7: Business days between dates

**Difficulty:** Medium · about 10 min · **Practises:** [`date`, `time`, `datetime`, `timedelta`](../learning/python-reference-09-dates-times-and-time-zones.ipynb#date-time-datetime-timedelta)

*SLAs are measured in working days, excluding weekends and holidays.*

Write `business_days(start, end, holidays=())` counting weekdays in `[start, end)` that are not in `holidays`.

**Sample calls**

| call | expected |
| --- | --- |
| `business_days(date(2026, 11, 23), date(2026, 11, 30))` | `5` |
| `business_days(date(2026, 11, 23), date(2026, 11, 30), [date(2026, 11, 26)])` | `4` |
| `business_days(date(2026, 11, 28), date(2026, 11, 30))` | `0` |

In [ ]:
def business_days(start, end, holidays=()):
    # your code here
    pass

In [ ]:
from datetime import timedelta

def business_days(start, end, holidays=()):
    holidays = set(holidays)
    count, day = 0, start
    while day < end:
        if day.weekday() < 5 and day not in holidays:
            count += 1
        day += timedelta(days=1)
    return count

## Exercise 8: Fill gaps in a daily series

**Difficulty:** Medium · about 10 min · **Practises:** [Month ends, adding months and date ranges](../learning/python-reference-09-dates-times-and-time-zones.ipynb#month-ends-adding-months-and-date-ranges)

*Dashboards need a row for every day, including days with no data.*

Write `fill_daily(points)` taking `{iso_date: value}` and returning a list of `(iso_date, value)` for every day from the earliest to the latest date, with 0 for missing days.

**Sample calls**

| call | expected |
| --- | --- |
| `fill_daily({"2026-11-01": 5, "2026-11-04": 2})` | `[('2026-11-01', 5), ('2026-11-02', 0), ('2026-11-03', 0), ('2026-11-04', 2)]` |
| `fill_daily({"2026-11-02": 1, "2026-11-01": 3})` | `[('2026-11-01', 3), ('2026-11-02', 1)]` |
| `fill_daily({})` | `[]` |

In [ ]:
def fill_daily(points):
    # your code here
    pass

In [ ]:
from datetime import date, timedelta

def fill_daily(points):
    if not points:
        return []
    days = sorted(date.fromisoformat(k) for k in points)
    out, day = [], days[0]
    while day <= days[-1]:
        out.append((day.isoformat(), points.get(day.isoformat(), 0)))
        day += timedelta(days=1)
    return out

## Exercise 9: Bucket events into 15-minute windows

**Difficulty:** Easy · about 5 min · **Practises:** [Month ends, adding months and date ranges](../learning/python-reference-09-dates-times-and-time-zones.ipynb#month-ends-adding-months-and-date-ranges)

*Tumbling windows for metrics, the batch version of a streaming window.*

Write `window_counts(isos, minutes=15)` returning a sorted list of `(window_start_iso, count)` for UTC ISO timestamps, flooring each to its window.

**Sample calls**

| call | expected |
| --- | --- |
| `window_counts(["2026-11-27T09:01:00Z", "2026-11-27T09:14:59Z", "2026-11-27T09:15:00Z"])` | `[('2026-11-27T09:00:00+00:00', 2), ('2026-11-27T09:15:00+00:00', 1)]` |
| `window_counts([])` | `[]` |
| `window_counts(["2026-11-27T09:59:00Z", "2026-11-27T10:00:00Z"], minutes=60)` | `[('2026-11-27T09:00:00+00:00', 1), ('2026-11-27T10:00:00+00:00', 1)]` |

In [ ]:
def window_counts(isos, minutes=15):
    # your code here
    pass

In [ ]:
from collections import Counter
from datetime import datetime, timedelta, timezone

def window_counts(isos, minutes=15):
    interval = timedelta(minutes=minutes)
    epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
    counts = Counter()
    for text in isos:
        dt = datetime.fromisoformat(text).astimezone(timezone.utc)
        counts[(dt - (dt - epoch) % interval).isoformat()] += 1
    return sorted(counts.items())

## Exercise 10: Epoch milliseconds

**Difficulty:** Easy · about 5 min · **Practises:** [Unix timestamps](../learning/python-reference-09-dates-times-and-time-zones.ipynb#unix-timestamps)

*Kafka and JavaScript timestamps are milliseconds since the epoch.*

Write `from_epoch_ms(ms)` returning a UTC ISO string with millisecond precision, and `to_epoch_ms(iso)` returning an int; a naive ISO string is treated as UTC.

**Sample calls**

| call | expected |
| --- | --- |
| `from_epoch_ms(1795771800123)` | `'2026-11-27T09:30:00.123+00:00'` |
| `to_epoch_ms("2026-11-27T09:30:00.123+00:00")` | `1795771800123` |
| `to_epoch_ms(from_epoch_ms(1700000000999))` | `1700000000999` |

In [ ]:
def from_epoch_ms(ms):
    # your code here
    pass


def to_epoch_ms(iso):
    # your code here
    pass

In [ ]:
from datetime import datetime, timedelta, timezone

EPOCH = datetime(1970, 1, 1, tzinfo=timezone.utc)

def from_epoch_ms(ms):
    return (EPOCH + timedelta(milliseconds=ms)).isoformat(timespec="milliseconds")

def to_epoch_ms(iso):
    dt = datetime.fromisoformat(iso)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return (dt - EPOCH) // timedelta(milliseconds=1)

## Exercise 11: Human-readable durations

**Difficulty:** Easy · about 5 min · **Practises:** [`date`, `time`, `datetime`, `timedelta`](../learning/python-reference-09-dates-times-and-time-zones.ipynb#date-time-datetime-timedelta)

*Run summaries print durations people can read.*

Write `humanize(seconds)` formatting a non-negative number of seconds as `1d 02h 03m 04s`, omitting leading zero units (`3m 05s`, `7s`); fractions are truncated.

**Sample calls**

| call | expected |
| --- | --- |
| `humanize(7)` | `'7s'` |
| `humanize(185)` | `'3m 05s'` |
| `humanize(3723)` | `'1h 02m 03s'` |

In [ ]:
def humanize(seconds):
    # your code here
    pass

In [ ]:
def humanize(seconds):
    total = int(seconds)
    days, rest = divmod(total, 86400)
    hours, rest = divmod(rest, 3600)
    minutes, secs = divmod(rest, 60)
    parts = [(days, "d"), (hours, "h"), (minutes, "m"), (secs, "s")]
    while len(parts) > 1 and parts[0][0] == 0:
        parts.pop(0)
    first, *others = parts
    return " ".join([f"{first[0]}{first[1]}"] + [f"{v:02d}{u}" for v, u in others])

## Exercise 12: ISO week labels

**Difficulty:** Easy · about 5 min · **Practises:** [`date`, `time`, `datetime`, `timedelta`](../learning/python-reference-09-dates-times-and-time-zones.ipynb#date-time-datetime-timedelta)

*Weekly reports keyed by ISO week, where the first days of January can belong to the previous year.*

Write `iso_week(d)` returning `YYYY-Www` (e.g. `2026-W48`) using the ISO year and week.

**Sample calls**

| call | expected |
| --- | --- |
| `iso_week(date(2026, 11, 27))` | `'2026-W48'` |
| `iso_week(date(2027, 1, 1))` | `'2026-W53'` |
| `iso_week(date(2026, 12, 31))` | `'2026-W53'` |

In [ ]:
def iso_week(d):
    # your code here
    pass

In [ ]:
def iso_week(d):
    year, week, _ = d.isocalendar()
    return f"{year}-W{week:02d}"

## Exercise 13: Cut-off times in local zones

**Difficulty:** Hard · about 15 min · **Practises:** [Naive and aware datetimes](../learning/python-reference-09-dates-times-and-time-zones.ipynb#naive-and-aware-datetimes), [Daylight saving time](../learning/python-reference-09-dates-times-and-time-zones.ipynb#daylight-saving-time)

*"Orders before 17:00 local time ship today": local business rules on UTC data.*

Write `ship_date(order_utc_iso, zone, cutoff_hour=17)` returning the local ISO date on which the order ships: the local order date if placed before the cutoff, else the next local business day (Mon–Fri); orders placed on a weekend ship the next Monday.

**Sample calls**

| call | expected |
| --- | --- |
| `ship_date("2026-11-25T20:00:00Z", "America/New_York")` | `'2026-11-25'` |
| `ship_date("2026-11-25T23:00:00Z", "America/New_York")` | `'2026-11-26'` |
| `ship_date("2026-11-27T22:30:00Z", "America/New_York")` | `'2026-11-30'` |

In [ ]:
def ship_date(order_utc_iso, zone, cutoff_hour=17):
    # your code here
    pass

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

def ship_date(order_utc_iso, zone, cutoff_hour=17):
    local = datetime.fromisoformat(order_utc_iso).astimezone(ZoneInfo(zone))
    day = local.date()
    if local.weekday() >= 5 or local.hour >= cutoff_hour:
        day += timedelta(days=1)
    while day.weekday() >= 5:
        day += timedelta(days=1)
    return day.isoformat()

**Next:** look anything up in [Python reference 9: dates, times and time zones](../learning/python-reference-09-dates-times-and-time-zones.ipynb), or try the [Python challenges](python-challenges-01-arrays-and-hashing.ipynb) and [Python coding problems](python-coding-problems.ipynb).